# Fine-tune SmolVLM2 on Pokemon TCG Cards (structured extraction)

Fine-tunes `HuggingFaceTB/SmolVLM2-500M-Video-Instruct` on `TheFusion21/PokemonCards` — given a card image, output `{"name": ..., "hp": ..., "set_name": ...}` as JSON.

Built for a **long unattended run on Kaggle (2x T4)**:
- Process pinned to a single GPU (avoids `Trainer` silently using `DataParallel` across both T4s).
- Model trained in **native fp16** (mixed-precision autocast crashes SmolVLM's `inputs_merger`).
- Trained with `max_steps` (not epochs) so run length is predictable, with frequent checkpointing (`save_steps`) so progress survives a restart, and auto-resume from the last checkpoint.
- **Sanity-check cell** that verifies the label collator isn't accidentally masking out all trainable tokens (this was silently broken in an earlier version — it produced `loss=0.0` / `nan` for the entire run).

In Kaggle: **Settings -> Accelerator -> GPU T4 x2**, and **Settings -> Internet -> On**. Use **Save Version -> Save & Run All (Commit)** for a run that survives you closing the browser tab.

In [ ]:
import os
# Restrict this process to a single GPU so Trainer doesn't wrap the model in
# DataParallel across both T4s (that causes cross-device tensor errors).
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install -q trl num2words wandb huggingface_hub

## Secrets: Weights & Biases + Hugging Face Hub

Reads `WANDB_API_KEY` and `HF_TOKEN` from **Kaggle Secrets** (Add-ons -> Secrets in the notebook editor) rather than hardcoding them. Neither key is ever written into this notebook's text or output.

In [ ]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

try:
    WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")
except Exception:
    WANDB_API_KEY = None
    print("[WARN] No WANDB_API_KEY secret found -- W&B logging will be skipped (set report_to='none' below).")

try:
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = None
    print("[WARN] No HF_TOKEN secret found -- the Hugging Face push cell will be skipped.")

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print("[INFO] Logged in to Weights & Biases.")

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, pipeline

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16  # native fp16 everywhere: T4 has no good bf16, and fp16 autocast crashes SmolVLM
MODEL_ID = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
print(f"[INFO] Using device: {DEVICE} | dtype: {DTYPE}")

## Load dataset

`TheFusion21/PokemonCards` only ships `image_url` (not embedded images), so we download each card image ourselves. We sample a subset (not all 13k rows) to keep download time and epoch count reasonable — increase `NUM_SAMPLES` if you want more data for a longer run.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("TheFusion21/PokemonCards")["train"]

NUM_SAMPLES = 1500  # increase for a longer/richer run
raw_dataset = raw_dataset.shuffle(seed=42).select(range(min(NUM_SAMPLES, len(raw_dataset))))
raw_dataset[0]

## Download images and format into chat message format

In [ ]:
import io
import json
import requests
from tqdm.auto import tqdm
from PIL import Image

SESSION = requests.Session()

USER_PROMPT = """This is a Pokemon TCG card. Reply with ONLY a raw JSON object:
{"name": "pokemon name", "hp": "HP digits only", "set_name": "the set name"}
"""

def download_image(url, timeout=10, retries=2):
    for attempt in range(retries):
        try:
            resp = SESSION.get(url, timeout=timeout)
            resp.raise_for_status()
            img = Image.open(io.BytesIO(resp.content)).convert("RGB")
            buf = io.BytesIO()
            img.save(buf, format="JPEG")
            return buf.getvalue()
        except Exception:
            if attempt == retries - 1:
                return None
    return None

def format_data(sample):
    img_bytes = download_image(sample["image_url"])
    if img_bytes is None:
        return None

    target = json.dumps({
        "name": sample["name"],
        "hp": str(sample["hp"]),
        "set_name": sample["set_name"],
    })

    # Every content dict has the same keys (type/text/image) so the schema
    # is consistent across rows, which avoids feature-inference errors.
    return {
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": img_bytes, "text": None},
                {"type": "text", "text": USER_PROMPT, "image": None},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": target, "image": None}]},
        ]
    }

dataset_processed = []
skipped = 0
for sample in tqdm(raw_dataset, desc="Downloading + formatting"):
    row = format_data(sample)
    if row is None:
        skipped += 1
        continue
    dataset_processed.append(row)

print(f"[INFO] Formatted {len(dataset_processed)} samples, skipped {skipped} (failed downloads)")
dataset_processed[0]

## Train / validation split

We build a `datasets.Dataset` (required by `SFTTrainer`), but let it **auto-infer** the schema instead of passing explicit `Features`. Explicit nested `Sequence`/`Features` for this image+text structure is unreliable across `datasets` versions — auto-inference of the raw-bytes image column is robust and works the same way regardless of version.

In [ ]:
import random
from datasets import Dataset

random.seed(42)
dataset_shuffled = random.sample(dataset_processed, k=len(dataset_processed))

train_split = round(0.85 * len(dataset_shuffled))
train_dataset = Dataset.from_list(dataset_shuffled[:train_split])
val_dataset = Dataset.from_list(dataset_shuffled[train_split:])

print(f"[INFO] Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

## Baseline: test the base (not fine-tuned) model

In [ ]:
import copy

# The pipeline needs a decoded PIL image, not raw bytes
example_base_model_input = copy.deepcopy(dataset_processed[0]["messages"][:1])
example_base_model_input[0]["content"][0]["image"] = Image.open(
    io.BytesIO(example_base_model_input[0]["content"][0]["image"])
)

pipe = pipeline(
    "image-text-to-text",
    model=MODEL_ID,
    device=DEVICE,
    dtype=DTYPE,
)

base_model_output = pipe(text=example_base_model_input, max_new_tokens=128)

print(f'[INFO] Base model output:\n{base_model_output[0]["generated_text"][-1]["content"]}\n')
print(f'[INFO] Desired output:\n{dataset_processed[0]["messages"][-1]["content"][0]["text"]}')

del pipe
torch.cuda.empty_cache()

## Load model for training

Model is loaded in native fp16 (`dtype=DTYPE`) and pinned to the single visible GPU. Mixed precision (`fp16=True` / `bf16=True` in the trainer config) is intentionally **not** used — it triggers a dtype-mismatch crash inside SmolVLM's `inputs_merger` and an "attempting to unscale FP16 gradients" error. Training natively in fp16 avoids both issues.

In [ ]:
model_kwargs = dict(
    attn_implementation="eager",
    dtype=DTYPE,
    device_map={"": 0},
)

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **model_kwargs)
processor = AutoProcessor.from_pretrained(MODEL_ID)

## Freeze the vision encoder (train only the language model parts)

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable, total - trainable

for param in model.model.vision_model.parameters():
    param.requires_grad = False

total_params, trainable_params, frozen_params = count_parameters(model)
print(f"[INFO] Total: {total_params:,} | Trainable: {trainable_params:,} | Frozen: {frozen_params:,}")

## Data collator

**This is the piece that was silently broken before.** The image placeholder token differs across processor versions/configs — hardcoding `"<image>"` can resolve to the wrong id and end up masking far more tokens than intended, leaving nothing for the model to learn from (this produced `loss=0.0000` / `eval_loss=nan` for an entire run previously). `get_image_token_id` looks up the id the processor actually uses.

In [ ]:
def get_image_token_id(processor):
    if getattr(processor, "image_token_id", None) is not None:
        return processor.image_token_id
    image_token = getattr(processor, "image_token", "<image>")
    if hasattr(image_token, "content"):
        image_token = image_token.content
    return processor.tokenizer.convert_tokens_to_ids(image_token)

IMAGE_TOKEN_ID = get_image_token_id(processor)
print(f"[INFO] Image placeholder token id: {IMAGE_TOKEN_ID}")

def convert_message_to_list_of_images(messages):
    image_inputs = []
    for msg in messages:
        content = msg.get("content", [])
        if not isinstance(content, list):
            content = [content]
        for element in content:
            if isinstance(element, dict) and element.get("type") == "image" and element.get("image") is not None:
                img = element["image"]
                if isinstance(img, (bytes, bytearray)):
                    img = Image.open(io.BytesIO(img))
                image_inputs.append(img.convert("RGB"))
    return image_inputs

def collate_fn(examples):
    texts, images = [], []
    for example in examples:
        images.append(convert_message_to_list_of_images(example["messages"]))
        text = processor.apply_chat_template(example["messages"], add_generation_prompt=False, tokenize=False)
        texts.append(text.strip())

    batch_input = processor(text=texts, images=images, return_tensors="pt", padding=True)

    labels = batch_input["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == IMAGE_TOKEN_ID] = -100
    batch_input["labels"] = labels

    return batch_input

## Sanity check: confirm the collator produces real training targets\n\nRun this **before** kicking off a multi-hour training job. If `valid_label_fraction` is near 0, something is wrong with the label masking and training will silently learn nothing (as happened before this fix).

In [ ]:
sample_batch = collate_fn([train_dataset[0], train_dataset[1]])
valid_label_fraction = (sample_batch["labels"] != -100).float().mean().item()
num_valid_tokens = int((sample_batch["labels"] != -100).sum().item())

print(f"[INFO] Batch label shape: {tuple(sample_batch['labels'].shape)}")
print(f"[INFO] Non-masked (trainable) label tokens: {num_valid_tokens}")
print(f"[INFO] Fraction of non-masked label tokens: {valid_label_fraction:.3f}")

assert num_valid_tokens > 0, "All labels are masked (-100) -- the collator is broken. Do not start training."
assert valid_label_fraction > 0.01, "Almost all labels are masked -- something is likely still wrong."
print("[INFO] Sanity check passed -- the collator has real training targets.")

## Training config

Uses `max_steps` instead of `num_train_epochs` so the run length is predictable regardless of dataset size or GPU speed. Checkpoints every `save_steps` so you can resume after any interruption. Adjust `MAX_STEPS` based on the observed `it/s` in the progress bar after the first few steps — e.g. if you see ~1 step every 20s, `600` steps is roughly 3.3 hours; scale up for a longer run.

In [ ]:
from trl import SFTConfig

MAX_STEPS = 900  # adjust based on observed step speed to hit your target run length

training_args = SFTConfig(
    output_dir="/kaggle/working/smolvlm2-pokemon-cards-v1",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,    # low, to fit fp16 training on a 16GB T4
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,   # effective batch size 16
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    learning_rate=2e-4,
    fp16=False,        # model is already native fp16 -- no autocast/GradScaler needed
    bf16=False,
    max_grad_norm=1.0,
    lr_scheduler_type="constant",
    push_to_hub=False,
    report_to="wandb" if WANDB_API_KEY else "none",
    run_name="smolvlm2-pokemon-cards-v1",
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    remove_unused_columns=False,
)

## Train

Automatically resumes from the latest checkpoint in `output_dir` if one exists (e.g. after a Kaggle session restart), instead of starting over. Note: `/kaggle/working` persists for the life of the interactive session, but a full session restart resets it unless you've committed a version — for real resilience across restarts, use **Save Version -> Save & Run All**.

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor,
    data_collator=collate_fn,
)

last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    checkpoints = [d for d in os.listdir(training_args.output_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        last_checkpoint = os.path.join(
            training_args.output_dir,
            sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1],
        )
        print(f"[INFO] Resuming from checkpoint: {last_checkpoint}")

trainer.train(resume_from_checkpoint=last_checkpoint)

In [ ]:
trainer.save_model()

## Push to Hugging Face Hub

Creates a model repo under your account and pushes the fine-tuned weights + processor. Uses `HF_TOKEN` from Kaggle Secrets — nothing is hardcoded. Skips cleanly if the secret isn't set.

In [ ]:
if HF_TOKEN:
    from huggingface_hub import HfApi, create_repo

    api = HfApi(token=HF_TOKEN)
    hf_username = api.whoami()["name"]
    REPO_ID = f"{hf_username}/smolvlm2-pokemon-cards-v1"

    create_repo(REPO_ID, token=HF_TOKEN, exist_ok=True, repo_type="model")
    print(f"[INFO] Repo ready: https://huggingface.co/{REPO_ID}")

    model.push_to_hub(REPO_ID, token=HF_TOKEN, commit_message="Fine-tuned SmolVLM2 on Pokemon TCG cards")
    processor.push_to_hub(REPO_ID, token=HF_TOKEN, commit_message="Add processor")

    print(f"[INFO] Push complete: https://huggingface.co/{REPO_ID}")
else:
    print("[INFO] Skipping Hugging Face push -- no HF_TOKEN secret found.")

## Plot training / validation loss

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_losses = [log["loss"] for log in log_history if "loss" in log]
step_train = [log["step"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
step_eval = [log["step"] for log in log_history if "eval_loss" in log]

plt.plot(step_train, train_losses, label="Training Loss")
plt.plot(step_eval, eval_losses, label="Validation Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

## Compare fine-tuned model vs base model

In [ ]:
CHECKPOINT_DIR_NAME = training_args.output_dir

del model, trainer
gc.collect()
torch.cuda.empty_cache()

pt_pipe = pipeline("image-text-to-text", model=MODEL_ID, dtype=DTYPE, device=DEVICE)
pt_pipe.model.generation_config.do_sample = False
pt_pipe.model.generation_config.pad_token_id = processor.tokenizer.eos_token_id

ft_pipe = pipeline("image-text-to-text", model=CHECKPOINT_DIR_NAME, dtype=DTYPE, device=DEVICE)
ft_pipe.model.generation_config.do_sample = False
ft_pipe.model.generation_config.pad_token_id = processor.tokenizer.eos_token_id

In [ ]:
random_val_sample = random.choice(val_dataset)
random_val_sample_model_input = dict(random_val_sample["messages"][0])
random_val_sample_model_input["content"] = [dict(c) for c in random_val_sample_model_input["content"]]
random_val_sample_image = Image.open(io.BytesIO(random_val_sample_model_input["content"][0]["image"]))
random_val_sample_model_input["content"][0]["image"] = random_val_sample_image  # decode bytes -> PIL for the pipeline
ideal_output_text = random_val_sample["messages"][-1]["content"][0]["text"]

print(f"[INFO] Ideal output:\n{ideal_output_text}\n")

pt_pipe_output = pt_pipe(text=[random_val_sample_model_input], max_new_tokens=128)
print(f"[INFO] Base model output:\n{pt_pipe_output[0]['generated_text'][-1]['content']}\n")

ft_pipe_output = ft_pipe(text=[random_val_sample_model_input], max_new_tokens=128)
print(f"[INFO] Fine-tuned model output:\n{ft_pipe_output[0]['generated_text'][-1]['content']}\n")

random_val_sample_image.resize(size=(400, 560))